# COUNT and COUNT DISTINCT using Warehouse Dataset

## Activity Overview

Spreadsheets and SQL have a lot in common. In a spreadsheet, the COUNT function is used to count the number of cells that contain numerical values in a specified range or in an array of cells. In SQL, COUNT and COUNT DISTINCT are similar tools. 

The COUNT function returns the number of records that are returned by a query. COUNT DISTINCT performs the same function as COUNT, but it also removes both duplicate rows of the same data and null values from the result set. In this notebook, we will use both COUNT and COUNT DISTINCT in our queries.

## Objective
- Use COUNT and COUNT DISTINCT in our queries to determine the amounts of things. 

COUNT and COUNT DISTINCT will return numerical values found within a dataset, helping us answer questions like, “How many customers did this?” Or, “How many transactions were there this month?” Or, “How many dates are in this dataset?”

## Step 1: Create Table
Here we use local csv files, so I use DuckDB for easier import. 
Let's name our table "orders" and "warehouse"

In [1]:
!pip install duckdb --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [2]:
import duckdb

con = duckdb.connect("warehouse_orders.duckdb")

In [ ]:
con.sql("""
CREATE TABLE warehouse AS
SELECT *
FROM read_csv_auto('WarehouseOrders_Warehouse.csv')
""")

In [ ]:
con.sql("""
CREATE TABLE orders AS
SELECT *
FROM read_csv_auto('WarehouseOrders_Orders.csv')
""")

## Step 2: Examine the warehouse table
In this scenario, we are junior data analysts working for a company that manufactures socks. We have access to data on the company’s customers, orders, warehouses, and products. Within the dataset, there are two tables: warehouse and orders. Begin by examining the warehouse table:

In [4]:
con.sql("""
SELECT *
FROM warehouse
LIMIT 10;
""")

┌──────────────┬──────────────────────────────┬──────────────────┬────────────────┬─────────┐
│ warehouse_id │       warehouse_alias        │ maximum_capacity │ employee_total │  state  │
│    int64     │           varchar            │      int64       │     int64      │ varchar │
├──────────────┼──────────────────────────────┼──────────────────┼────────────────┼─────────┤
│         1543 │ Somerset Fulfillment Center  │              210 │             14 │ KY      │
│         2270 │ Bowling Green Warehouse      │              280 │             13 │ KY      │
│         2666 │ Lansing Fulfillment Center   │              290 │             16 │ MI      │
│         3417 │ Gatlinburg Warehouse         │              620 │              6 │ TN      │
│         3961 │ Lansing Storage Warehouse    │              740 │             22 │ MI      │
│         4338 │ Knoxville Fulfillment Center │              215 │             13 │ TN      │
│         6509 │ Memphis Fulfillment Center   │             

After running the query, the five columns from the warehouse table will load in the Query results window:

- warehouse_id: indicates the ID number of each warehouse 
- warehouse_alias: indicates the alias, or name, of each warehouse
- maximum_capacity: indicates the maximum capacity at each warehouse
- employee_total: indicates the total number of employees at each warehouse
- state: indicates the postal abbreviation for the U.S. state each warehouse is located in

## Step 3: Examine the orders table

In [5]:
con.sql("""
SELECT *
FROM orders
LIMIT 10;
""")

┌──────────┬─────────────┬──────────────┬────────────┬──────────────┐
│ order_id │ customer_id │ warehouse_id │ order_date │ shipper_date │
│  int64   │    int64    │    int64     │    date    │     date     │
├──────────┼─────────────┼──────────────┼────────────┼──────────────┤
│      789 │        3731 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      790 │        3486 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      791 │        2623 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      792 │        9869 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      793 │        6866 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      794 │        8055 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      795 │        1152 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      796 │        5765 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      797 │        6709 │         8118 │ 2019-01-01 │ 2019-01-04   │
│      798 │        4866 │         2666 │ 2019-01-01 │ 2019-01-04   │
└──────────┴────────

After running the query, the five columns from the orders table will load in the Query results window:

- order_id: indicates the ID number of each order
- customer_id: indicates the ID number of each customer
- warehouse_id: indicates the ID number of each warehouse
- order_date: indicates the date on which the order was placed
- shipper_date: indicates the date on which the order was shipped

## Step 4: JOIN the tables

Perhaps we need both the warehouse details and the order details so we can report on the distribution of orders by state. Use JOIN to join the two tables together to get data from both of them and alias the warehouse table in the process. In this case, let's use JOIN as shorthand for INNER JOIN to get corresponding data from both tables. 

In [6]:
con.sql("""
SELECT
	orders.*,
	warehouse.warehouse_alias,
	warehouse.state
FROM
	orders
JOIN
    warehouse ON orders.warehouse_id = warehouse.warehouse_id
""")

┌──────────┬─────────────┬──────────────┬────────────┬──────────────┬──────────────────────────────┬─────────┐
│ order_id │ customer_id │ warehouse_id │ order_date │ shipper_date │       warehouse_alias        │  state  │
│  int64   │    int64    │    int64     │    date    │     date     │           varchar            │ varchar │
├──────────┼─────────────┼──────────────┼────────────┼──────────────┼──────────────────────────────┼─────────┤
│      789 │        3731 │         8118 │ 2019-01-01 │ 2019-01-04   │ Ann Arbor Fulfillment Center │ MI      │
│      790 │        3486 │         8118 │ 2019-01-01 │ 2019-01-04   │ Ann Arbor Fulfillment Center │ MI      │
│      791 │        2623 │         8118 │ 2019-01-01 │ 2019-01-04   │ Ann Arbor Fulfillment Center │ MI      │
│      792 │        9869 │         8118 │ 2019-01-01 │ 2019-01-04   │ Ann Arbor Fulfillment Center │ MI      │
│      793 │        6866 │         8118 │ 2019-01-01 │ 2019-01-04   │ Ann Arbor Fulfillment Center │ MI      │
│

After running the query, the data from both tables are now joined together in the Query results window with these seven columns:

- order_id: indicates the ID number of each order.
- customer_id: indicates the ID number of each customer.
- warehouse_id: indicates the ID number of each warehouse.
- order_date: indicates the date on which the order was placed.
- shipper_date: indicates the date on which the order was shipped.
- warehouse_alias: indicates the names given to each warehouse as an alias.
- state: indicates which state the warehouse is located in.

## Step 5: Find the number of states with warehouses that shipped orders with COUNT DISTINCT

As a data analyst, we might be interested in finding the number of states with warehouses that have shipped orders. 

First, we'll try to do this with the COUNT function. 

In [7]:
con.sql("""
SELECT
	COUNT(warehouse.state) as num_states
FROM
	orders
JOIN
    warehouse ON orders.warehouse_id = warehouse.warehouse_id
""")

┌────────────┐
│ num_states │
│   int64    │
├────────────┤
│       9999 │
└────────────┘

The query returned more than 9,000 results. There are only 50 states, so this is clearly not the answer we're looking for. \
This is because the query counted every single record (row) that included a state, regardless of duplicates or null values. 

We can modify the existing query to remove duplicates and null values, and only count the distinct states with <b>COUNT DISTINCT</b>,it will remove all the repeated instances from the results. 

In [8]:
con.sql("""
SELECT
	COUNT(DISTINCT warehouse.state) as num_states
FROM
	orders
JOIN
    warehouse ON orders.warehouse_id = warehouse.warehouse_id
""")

┌────────────┐
│ num_states │
│   int64    │
├────────────┤
│          3 │
└────────────┘

According to the results, there are three distinct states in the orders data.

## Step 6: Use GROUP BY to group the number of orders by state

Next, we might want to find the number of orders shipped from warehouses in each state, instead of the number that shipped orders. We can find this information by using GROUP BY to group the state column in the warehouse table. Use JOIN and GROUP BY in the FROM statement. 

In [9]:
con.sql("""
SELECT
	state,
	COUNT(DISTINCT order_id) as num_orders
FROM
	orders
JOIN
    warehouse ON orders.warehouse_id = warehouse.warehouse_id
GROUP BY
	warehouse.state
""")

┌─────────┬────────────┐
│  state  │ num_orders │
│ varchar │   int64    │
├─────────┼────────────┤
│ KY      │       1048 │
│ MI      │       6205 │
│ TN      │       2746 │
└─────────┴────────────┘

After running the query, there are now three rows listed in the results table: one for each state represented within the orders data. The Query results window now displays two columns:
- Column one is state, which  indicates which state the warehouse is located in. 
- Column two is num_orders, which indicates the number of orders.\
These three numbers add up to the count that we ran previously—9,999. 

Now we have successfully executed queries using COUNT and COUNT DISTINCT, as well as SELECT, FROM, and GROUP BY statements to create aliases and JOIN tables, return numerical values within a specific range, and group by specific columns within a table. 